# Anomaly Detection

## 0. Imports and data

We work with NovaShop's daily KPIs. One observation = **one day**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
df = pd.read_csv("novashop_daily_operations.csv", parse_dates=["date"])
print(df.shape)
df.head()

In [ ]:
df.info()

## 1. First, explore the data

Before running a detector, let's examine several KPIs over time.

In [ ]:
cols = ["orders", "payment_failure_rate", "site_latency_ms", "return_rate"]
for col in cols:
    plt.figure(figsize=(11, 3))
    plt.plot(df["date"], df[col])
    plt.title(col)
    plt.xlabel("date")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## 2. LOF in two dimensions

We start with two features to visualize the geometry of the problem. LOF relies on distances, so **we scale the features**.

In [ ]:
features_2d = ["orders", "site_latency_ms"]
X2 = df[features_2d].copy()
scaler_2d = StandardScaler()
X2_scaled = scaler_2d.fit_transform(X2)

plt.figure(figsize=(7, 5))
plt.scatter(X2_scaled[:, 0], X2_scaled[:, 1], alpha=0.65)
plt.xlabel("orders - standardized")
plt.ylabel("site_latency_ms - standardized")
plt.title("NovaShop: days as points")
plt.show()

In [ ]:
lof_2d = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_label_2d = lof_2d.fit_predict(X2_scaled)

# sklearn stores negative_outlier_factor_: more negative values indicate stronger outliers.
# Create an interpretable score: higher values indicate more unusual observations.
lof_score_2d = -lof_2d.negative_outlier_factor_

result_2d = df[["date"] + features_2d].copy()
result_2d["lof_score"] = lof_score_2d
result_2d["lof_anomaly"] = lof_label_2d == -1
result_2d.sort_values("lof_score", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(7, 5))
normal = ~result_2d["lof_anomaly"]
anom = result_2d["lof_anomaly"]
plt.scatter(X2_scaled[normal, 0], X2_scaled[normal, 1], alpha=0.55, label="other observations")
plt.scatter(X2_scaled[anom, 0], X2_scaled[anom, 1], s=80, marker="x", label="LOF: flagged")
plt.xlabel("orders - standardized")
plt.ylabel("site_latency_ms - standardized")
plt.title("LOF: results in 2D")
plt.legend()
plt.show()

### Interpretation
1. Are all flagged days simply the largest values on the X or Y axis?
2. Can you find a point whose unusualness results from **its position relative to its neighbors**?

## 3. What changes when we vary the number of neighbors?

We are not looking for the 'best k'. We want to see how changing the neighborhood changes the definition of locality.

In [ ]:
lof_k = pd.DataFrame({"date": df["date"]})
for k in [5, 10, 20, 40]:
    model = LocalOutlierFactor(n_neighbors=k, contamination=0.02)
    model.fit_predict(X2_scaled)
    lof_k[f"score_k{k}"] = -model.negative_outlier_factor_

lof_k.head()

In [ ]:
for col in ["score_k5", "score_k10", "score_k20", "score_k40"]:
    top = lof_k.nlargest(5, col)[["date", col]]
    print(f"\nTOP 5 for {col}")
    print(top.to_string(index=False))

**Question:** Which days consistently rank high regardless of `n_neighbors`, and which appear only when using a very local perspective?

## 4. Isolation Forest - a different definition of unusualness

Now we use more features simultaneously. Unlike LOF, this model does not compare local densities.

In [ ]:
features = [
    "orders",
    "conversion_rate",
    "avg_order_value",
    "return_rate",
    "payment_failure_rate",
    "site_latency_ms",
    "support_tickets",
]

X = df[features].copy()
X.describe().T

In [ ]:
iso = IsolationForest(
    n_estimators=300,
    contamination=0.02,
    random_state=42
)
iso.fit(X)

# score_samples: higher values indicate more normal observations. We reverse the sign for consistency.
df_result = df.copy()
df_result["if_anomaly_score"] = -iso.score_samples(X)
df_result["if_anomaly"] = iso.predict(X) == -1

df_result.sort_values("if_anomaly_score", ascending=False)[
    ["date", "if_anomaly_score", "if_anomaly"] + features
].head(12)

### Do not look only at the flag
First, examine the ranking. For the 3-5 highest-scoring days, try to explain **what exactly makes them unusual**.

In [ ]:
top_if = df_result.nlargest(10, "if_anomaly_score")
top_if[["date", "if_anomaly_score"] + features]

## 5. `contamination`: the data stay the same, but the threshold changes

Compare the number of flagged observations for the same dataset.

In [ ]:
contamination_results = []
for contamination in [0.01, 0.02, 0.05, 0.10]:
    m = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )
    labels = m.fit_predict(X)
    contamination_results.append({
        "contamination": contamination,
        "n_flagged": int((labels == -1).sum())
    })

pd.DataFrame(contamination_results)

## 6. LOF and Isolation Forest using the same feature representation

For a fairer comparison, both models receive the same set of features. We scale the features for LOF.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_labels = lof.fit_predict(X_scaled)

comparison = df[["date"]].copy()
comparison["lof_score"] = -lof.negative_outlier_factor_
comparison["lof_anomaly"] = lof_labels == -1
comparison["if_score"] = df_result["if_anomaly_score"]
comparison["if_anomaly"] = df_result["if_anomaly"]
comparison["agreement"] = comparison["lof_anomaly"] == comparison["if_anomaly"]

comparison.sort_values(["lof_anomaly", "if_anomaly"], ascending=False).head(15)

In [ ]:
pd.crosstab(
    comparison["lof_anomaly"],
    comparison["if_anomaly"],
    rownames=["LOF"],
    colnames=["Isolation Forest"]
)

### Discussion
- Which days are flagged by both models?
- Which are flagged by only one model?
- Does disagreement mean that one model is wrong?
- What characteristics do the cases of disagreement have?

In [ ]:
flagged = comparison[comparison["lof_anomaly"] | comparison["if_anomaly"]]
flagged.merge(df[["date"] + features], on="date").sort_values(
    ["lof_anomaly", "if_anomaly"], ascending=False
)

## 7. Context: `campaign_active` and `is_weekend`

A detector may flag a statistically unusual day that makes sense once context is considered.

In [ ]:
flagged_context = flagged.merge(
    df[["date", "campaign_active", "is_weekend"] + features],
    on="date"
)
flagged_context.sort_values("if_score", ascending=False)

**Question:** Should a campaign day automatically be treated as a 'problem'? What changes when you know that an observation comes from Black Friday or another event?

# 8. Revealing the `ground truth`

In [ ]:
key = pd.read_csv("INSTRUCTOR_anomaly_key.csv", parse_dates=["date"])
key[key["known_anomaly"] == 1]

In [ ]:
evaluation = comparison.merge(key, on="date")
known = evaluation[evaluation["known_anomaly"] == 1].merge(
    df[["date"] + features + ["campaign_active", "is_weekend"]], on="date"
)
known[[
    "date", "anomaly_type", "lof_anomaly", "if_anomaly",
    "lof_score", "if_score", "campaign_active", "is_weekend"
] + features]

In [ ]:
summary = pd.DataFrame({
    "model": ["LOF", "Isolation Forest"],
    "flagged_total": [evaluation["lof_anomaly"].sum(), evaluation["if_anomaly"].sum()],
    "known_anomalies_detected": [
        ((evaluation["lof_anomaly"]) & (evaluation["known_anomaly"] == 1)).sum(),
        ((evaluation["if_anomaly"]) & (evaluation["known_anomaly"] == 1)).sum(),
    ]
})
summary